In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [3]:
len(ground_truth)

395

In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [5]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [6]:
q = ground_truth[10]
q

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [7]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
)

In [10]:
q['question']

'How do I join the Office Hours or live workshop if I don’t have the Zoom link?'

In [11]:
answer = assistant.rag(q['question'])

In [12]:
assistant.total_cost()

0.001164

In [13]:
print(answer)

Students don’t use the Zoom link directly. Join the session via **YouTube Live**, and submit questions through **Slido** (the link is pinned in chat when live). The live video URL is usually posted in the **announcements channel on Telegram and Slack** before the session starts, and you can also watch on the **DataTalksClub YouTube channel**.




In [14]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [15]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': 'Students don’t use the Zoom link directly. Join the session via **YouTube Live**, and submit questions through **Slido** (the link is pinned in chat when live). The live video URL is usually posted in the **announcements channel on Telegram and Slack** before the session starts, and you can also watch on the **DataTalksClub YouTube channel**.\n\n',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c9d9'}

In [16]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }
    print(result)

    return result

In [17]:
record = generate_rag_answer(q)
record

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?', 'answer_llm': 'You can join via **YouTube Live** even without the Zoom link. The Zoom link is only shared with instructors/presenters/TAs.\n\nCheck the **announcements channel on Telegram and Slack** for the live video URL before it starts, or watch on the DataTalksClub **YouTube Channel**. Questions should be submitted in **Slido** (linked in the live chat), not in the chat itself.', 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.', 'document':

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': 'You can join via **YouTube Live** even without the Zoom link. The Zoom link is only shared with instructors/presenters/TAs.\n\nCheck the **announcements channel on Telegram and Slack** for the live video URL before it starts, or watch on the DataTalksClub **YouTube Channel**. Questions should be submitted in **Slido** (linked in the live chat), not in the chat itself.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'documen

In [18]:
assistant.total_cost()

0.0023685

In [19]:
assistant.reset_usage()

In [20]:
assistant.total_cost()

0.0

In [21]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [22]:
with ThreadPoolExecutor(max_workers=10) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/395 [00:00<?, ?it/s]

{'question': 'Can I still take this course even if I missed the start date?', 'answer_llm': 'Yes, you can still join the course even if you missed the start date. You can start whenever you want.\n\nIf you want a certificate, make sure to submit your project while submissions are still being accepted.', 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'document': '74eb249bbf'}
{'question': 'If I filled out the registration form, does that mean I’m officially on a checked list for the course?', 'answer_llm': 'No. Registration is just to gauge interest, and it is not checked against any registered list. You’re accepted, and you can start learning and submitting homework while the form is open.', 'answer_orig': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Regist

KeyboardInterrupt: 

{'question': 'Can I use `client.responses.create(...)` and `response.output` with Mistral, Groq, or Gemini in my agent code?', 'answer_llm': 'No. `client.responses.create(...)` and `response.output` are OpenAI-specific, so they don’t work with Mistral, Groq, Gemini, etc.\n\nFor those providers, use the chat-completions API instead, and read:\n\n```python\nresponse.choices[0].message.content\n```\n\nExample:\n\n```python\nresponse = client.chat.completions.create(\n    model="<provider-model>",\n    messages=[{"role": "user", "content": prompt}],\n    tools=tools_schema,\n)\nreturn response.choices[0].message.content\n```\n\nYou may also need to adapt the tool schema to the provider’s expected format.', 'answer_orig': 'The new OpenAI Responses API (`client.responses.create(...)`, accessed via `response.output`) is OpenAI-specific. Other providers (Mistral, Groq, Gemini, etc.) don\'t implement it.\n\nFor non-OpenAI providers, use the chat-completions API and read `response.choices[0].mes

{'question': "What should I use instead of the Responses API if I'm switching my LLM provider away from OpenAI?", 'answer_llm': 'Use the **chat-completions API** instead of the Responses API.\n\nFor non-OpenAI providers, read the answer from:\n\n```python\nresponse.choices[0].message.content\n```\n\nExample:\n\n```python\nresponse = client.chat.completions.create(\n    model="<provider-model>",\n    messages=[{"role": "user", "content": prompt}],\n    tools=tools_schema,  # may need adapting per provider\n)\nreturn response.choices[0].message.content\n```\n\nYou may also need to **adapt the tools schema** to match what your provider expects.', 'answer_orig': 'The new OpenAI Responses API (`client.responses.create(...)`, accessed via `response.output`) is OpenAI-specific. Other providers (Mistral, Groq, Gemini, etc.) don\'t implement it.\n\nFor non-OpenAI providers, use the chat-completions API and read `response.choices[0].message.content`:\n\n```python\nresponse = client.chat.completi

In [ ]:
results[:10]

[{'question': 'Is it okay to join the course late if I just found it now?',
  'answer_llm': 'Yes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'answer_llm': 'Yes, you can still join if you missed the start date, but if you want a certificate, you need to finish with the live cohort and submit your project while submissions are still open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still eligible for a certificate?',
  'answer_llm': 'Yes, as long 

In [ ]:
df_results = pd.DataFrame(results)

NameError: name 'results' is not defined

In [ ]:
df_results.head()

,question,answer_llm,answer_orig,document
0,Is it okay to join the course late if I just f...,"Yes, you can still join the course late. If yo...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still take this course even if I missed ...,"Yes, you can still join if you missed the star...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,If I join after the course has already started...,"Yes, as long as you join while the course is s...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,Do I need to submit my project before submissi...,"Yes — to get the certificate, you need to subm...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,I’m a bit late to the course—what do I need to...,"To still earn the certificate, you need to:\n\...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [ ]:
assistant.total_cost()


0.34332825

In [ ]:
df_results.to_csv("data/rag-answers-new.csv", index=False)